In [1]:
# ─────────────────────────────────────────────────────────────
# 03 SIEM RULES — LANL Authentication Dataset
# Rules grounded in statistically validated evidence from
# 02_features.ipynb, not assumed thresholds
# ─────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

df = pd.read_csv('df_with_features.csv')

print(f"Loaded {len(df):,} events")
print(f"Attack ratio: {df['is_attack'].mean()*100:.2f}%")

# ── Test logon_type discriminative power ─────────────────────
print("\nlogon_type distribution — attack accounts:")
print(df[df['is_attack']==1]['logon_type'].value_counts(normalize=True).head(10) * 100)

print("\nlogon_type distribution — normal accounts:")
print(df[df['is_attack']==0]['logon_type'].value_counts(normalize=True).head(10) * 100)

# Chi-square test for independence
contingency = pd.crosstab(df['is_attack'], df['logon_type'])
chi2, p_value, dof, expected = chi2_contingency(contingency)
print(f"\nChi-square statistic: {chi2:.4f}")
print(f"p-value: {p_value:.6f}")
print(f"Significant at p<0.05: {p_value < 0.05}")

Loaded 68,221 events
Attack ratio: 14.98%

logon_type distribution — attack accounts:
logon_type
Network              70.893259
?                    27.482634
Unlock                0.792486
Batch                 0.547892
RemoteInteractive     0.156540
Interactive           0.088054
CachedInteractive     0.029351
Service               0.009784
Name: proportion, dtype: float64

logon_type distribution — normal accounts:
logon_type
Network              83.137931
?                    14.756897
Service               1.272414
Unlock                0.272414
NewCredentials        0.163793
Interactive           0.156897
Batch                 0.153448
NetworkCleartext      0.046552
CachedInteractive     0.032759
RemoteInteractive     0.006897
Name: proportion, dtype: float64

Chi-square statistic: 1354.0558
p-value: 0.000000
Significant at p<0.05: True


In [2]:
# Test RemoteInteractive specifically
df['is_remote_interactive'] = (df['logon_type'] == 'RemoteInteractive').astype(int)
contingency_ri = pd.crosstab(df['is_attack'], df['is_remote_interactive'])
chi2_ri, p_ri, _, _ = chi2_contingency(contingency_ri)
print("RemoteInteractive logon type:")
print(contingency_ri)
print(f"Chi-square: {chi2_ri:.4f}, p-value: {p_ri:.6f}\n")

# Test Batch specifically
df['is_batch'] = (df['logon_type'] == 'Batch').astype(int)
contingency_batch = pd.crosstab(df['is_attack'], df['is_batch'])
chi2_batch, p_batch, _, _ = chi2_contingency(contingency_batch)
print("Batch logon type:")
print(contingency_batch)
print(f"Chi-square: {chi2_batch:.4f}, p-value: {p_batch:.6f}")

RemoteInteractive logon type:
is_remote_interactive      0   1
is_attack                       
0                      57996   4
1                      10205  16
Chi-square: 61.3875, p-value: 0.000000

Batch logon type:
is_batch       0   1
is_attack           
0          57911  89
1          10165  56
Chi-square: 61.8989, p-value: 0.000000


In [4]:
# ═══════════════════════════════════════════════════════════
# SIEM RULE DEFINITIONS — LANL Dataset
# Each rule grounded in statistically validated evidence
# from feature engineering and the diagnostics above
# ═══════════════════════════════════════════════════════════

alerts = []

# ── Rule 1: Excessive Lateral Movement (T1021 — Remote Services) ──
# STRONGEST signal in this dataset: attack accounts touch 5x more
# distinct destination computers (20.24 vs 4.16, p<0.000001)
LATERAL_MOVEMENT_THRESHOLD = 10  # above the 75th percentile (7) but below attack mean (20.24)

rule1 = df[df['unique_destination_computers'] >= LATERAL_MOVEMENT_THRESHOLD]
for _, row in rule1.iterrows():
    alerts.append({
        'time': row['time'],
        'rule': 'Excessive Lateral Movement',
        'severity': 'HIGH',
        'user': row['source_user'],
        'computer': row['destination_computer'],
        'detail': f"Distinct destinations: {row['unique_destination_computers']}"
    })

# ── Rule 2: Suspicious Remote Logon Type (T1021.001 — RDP) ──
# RemoteInteractive logons are 22x more common in attack accounts
# (16/10221 vs 4/58000, chi2=61.39, p<0.000001)
rule2 = df[df['logon_type'] == 'RemoteInteractive']
for _, row in rule2.iterrows():
    alerts.append({
        'time': row['time'],
        'rule': 'Suspicious Remote Logon Type',
        'severity': 'HIGH',
        'user': row['source_user'],
        'computer': row['destination_computer'],
        'detail': f"LogonType: {row['logon_type']}"
    })

# ── Rule 3: Batch/Scripted Authentication (T1059 — Scripting) ──
# Batch logons are 3.6x more common in attack accounts
# (56/10221 vs 89/58000, chi2=61.90, p<0.000001)
rule3 = df[df['logon_type'] == 'Batch']
for _, row in rule3.iterrows():
    alerts.append({
        'time': row['time'],
        'rule': 'Batch Scripted Authentication',
        'severity': 'MEDIUM',
        'user': row['source_user'],
        'computer': row['destination_computer'],
        'detail': f"LogonType: {row['logon_type']}"
    })

# ── Rule 4: Anomalously Clean Authentication at Scale ──
# REFRAMED from "High Failure Rate" based on evidence:
# attackers show LOWER failure rate but HIGHER volume than normal
# users — flags accounts with perfect/near-perfect success AND
# above-median login velocity, consistent with stolen valid
# credentials used persistently (T1078 — Valid Accounts)
CLEAN_AUTH_VELOCITY_THRESHOLD = df['logins_per_hour'].median()  # 2.0

rule4 = df[
    (df['is_failed_login'] == 0) &
    (df['logins_per_hour'] >= CLEAN_AUTH_VELOCITY_THRESHOLD) &
    (df['unique_destination_computers'] >= 5)  # combine with some lateral spread
]
for _, row in rule4.iterrows():
    alerts.append({
        'time': row['time'],
        'rule': 'Anomalously Clean Authentication at Scale',
        'severity': 'MEDIUM',
        'user': row['source_user'],
        'computer': row['destination_computer'],
        'detail': f"Velocity: {row['logins_per_hour']}, Destinations: {row['unique_destination_computers']}"
    })

alerts_df = pd.DataFrame(alerts)
alerts_df.to_csv('siem_alerts.csv', index=False)

print(f"Total SIEM alerts fired: {len(alerts_df):,}")
print(f"\nAlerts by rule:")
print(alerts_df['rule'].value_counts())
print(f"\nAlerts by severity:")
print(alerts_df['severity'].value_counts())

Total SIEM alerts fired: 35,183

Alerts by rule:
rule
Anomalously Clean Authentication at Scale    19609
Excessive Lateral Movement                   15409
Batch Scripted Authentication                  145
Suspicious Remote Logon Type                    20
Name: count, dtype: int64

Alerts by severity:
severity
MEDIUM    19754
HIGH      15429
Name: count, dtype: int64


In [5]:
ATTACK_ACCOUNTS = set(df[df['is_attack']==1]['source_user'].unique())

print("Precision per rule (% of alerts that are genuine attack accounts):")
print("=" * 60)

for rule_name in alerts_df['rule'].unique():
    rule_alerts = alerts_df[alerts_df['rule'] == rule_name]
    is_attack_alert = rule_alerts['user'].isin(ATTACK_ACCOUNTS)
    precision = is_attack_alert.mean()
    print(f"\n{rule_name}")
    print(f"  Total alerts: {len(rule_alerts):,}")
    print(f"  True positives: {is_attack_alert.sum():,}")
    print(f"  Precision: {precision*100:.2f}%")

Precision per rule (% of alerts that are genuine attack accounts):

Excessive Lateral Movement
  Total alerts: 15,409
  True positives: 9,556
  Precision: 62.02%

Suspicious Remote Logon Type
  Total alerts: 20
  True positives: 16
  Precision: 80.00%

Batch Scripted Authentication
  Total alerts: 145
  True positives: 56
  Precision: 38.62%

Anomalously Clean Authentication at Scale
  Total alerts: 19,609
  True positives: 8,989
  Precision: 45.84%


In [6]:
print("Recall check — are we catching genuine attack ACCOUNTS (not just events)?")
print("=" * 60)

total_attack_accounts = len(ATTACK_ACCOUNTS)
print(f"Total distinct attack accounts in dataset: {total_attack_accounts}")

for rule_name in alerts_df['rule'].unique():
    rule_alerts = alerts_df[alerts_df['rule'] == rule_name]
    caught_accounts = set(rule_alerts['user']) & ATTACK_ACCOUNTS
    print(f"\n{rule_name}")
    print(f"  Distinct attack accounts caught: {len(caught_accounts)} / {total_attack_accounts}")

all_caught = set(alerts_df['user']) & ATTACK_ACCOUNTS
print(f"\nCombined — any rule caught: {len(all_caught)} / {total_attack_accounts} attack accounts")
missed = ATTACK_ACCOUNTS - all_caught
print(f"Missed entirely: {len(missed)} attack accounts")
if missed:
    print(f"Sample missed accounts: {list(missed)[:10]}")

Recall check — are we catching genuine attack ACCOUNTS (not just events)?
Total distinct attack accounts in dataset: 60

Excessive Lateral Movement
  Distinct attack accounts caught: 48 / 60

Suspicious Remote Logon Type
  Distinct attack accounts caught: 13 / 60

Batch Scripted Authentication
  Distinct attack accounts caught: 3 / 60

Anomalously Clean Authentication at Scale
  Distinct attack accounts caught: 50 / 60

Combined — any rule caught: 51 / 60 attack accounts
Missed entirely: 9 attack accounts
Sample missed accounts: ['U1789@DOM1', 'U86@C10', 'U7761@C2519', 'U8170@DOM1', 'U7004@C2519', 'U8168@C19038', 'U1519@DOM1', 'U737@C10', 'U1581@DOM1']


In [7]:
# Sweep lateral movement threshold to find a better operating point
print("Lateral Movement threshold sweep:")
print(f"{'Threshold':>10} {'Alerts':>10} {'Precision':>10} {'Accounts Caught':>18}")
print("-" * 52)

for threshold in [10, 15, 20, 25, 30, 35, 40]:
    subset = df[df['unique_destination_computers'] >= threshold]
    is_attack_alert = subset['source_user'].isin(ATTACK_ACCOUNTS)
    precision = is_attack_alert.mean() if len(subset) > 0 else 0
    accounts_caught = len(set(subset['source_user']) & ATTACK_ACCOUNTS)
    print(f"{threshold:>10} {len(subset):>10,} {precision*100:>9.2f}% {accounts_caught:>15}/60")

Lateral Movement threshold sweep:
 Threshold     Alerts  Precision    Accounts Caught
----------------------------------------------------
        10     15,409     62.02%              48/60
        15     12,064     65.95%              40/60
        20      5,564     93.46%              26/60
        25      2,427     90.65%              11/60
        30      1,027     77.90%               4/60
        35        600    100.00%               3/60
        40        400    100.00%               2/60


In [8]:
print("Clean Authentication at Scale — destination threshold sweep:")
print(f"{'Min Destinations':>18} {'Alerts':>10} {'Precision':>10} {'Accounts Caught':>18}")
print("-" * 60)

for dest_threshold in [5, 8, 10, 12, 15, 20]:
    subset = df[
        (df['is_failed_login'] == 0) &
        (df['logins_per_hour'] >= 2) &
        (df['unique_destination_computers'] >= dest_threshold)
    ]
    is_attack_alert = subset['source_user'].isin(ATTACK_ACCOUNTS)
    precision = is_attack_alert.mean() if len(subset) > 0 else 0
    accounts_caught = len(set(subset['source_user']) & ATTACK_ACCOUNTS)
    print(f"{dest_threshold:>18} {len(subset):>10,} {precision*100:>9.2f}% {accounts_caught:>15}/60")

Clean Authentication at Scale — destination threshold sweep:
  Min Destinations     Alerts  Precision    Accounts Caught
------------------------------------------------------------
                 5     19,609     45.84%              50/60
                 8     14,936     58.90%              49/60
                10     14,368     59.92%              48/60
                12     13,701     58.87%              45/60
                15     11,243     63.66%              40/60
                20      5,022     92.79%              26/60


In [9]:
alerts = []

# ── Rule 1: Excessive Lateral Movement (T1021) — threshold 20 ──
rule1 = df[df['unique_destination_computers'] >= 20]
for _, row in rule1.iterrows():
    alerts.append({
        'time': row['time'], 'rule': 'Excessive Lateral Movement',
        'severity': 'HIGH', 'user': row['source_user'],
        'computer': row['destination_computer'],
        'detail': f"Distinct destinations: {row['unique_destination_computers']}"
    })

# ── Rule 2: Suspicious Remote Logon Type (T1021.001) — unchanged ──
rule2 = df[df['logon_type'] == 'RemoteInteractive']
for _, row in rule2.iterrows():
    alerts.append({
        'time': row['time'], 'rule': 'Suspicious Remote Logon Type',
        'severity': 'HIGH', 'user': row['source_user'],
        'computer': row['destination_computer'],
        'detail': f"LogonType: {row['logon_type']}"
    })

# ── Rule 3: Batch Scripted Authentication (T1059) — unchanged ──
rule3 = df[df['logon_type'] == 'Batch']
for _, row in rule3.iterrows():
    alerts.append({
        'time': row['time'], 'rule': 'Batch Scripted Authentication',
        'severity': 'MEDIUM', 'user': row['source_user'],
        'computer': row['destination_computer'],
        'detail': f"LogonType: {row['logon_type']}"
    })

# ── Rule 4: Anomalously Clean Authentication at Scale (T1078) — threshold 15 ──
rule4 = df[
    (df['is_failed_login'] == 0) &
    (df['logins_per_hour'] >= 2) &
    (df['unique_destination_computers'] >= 15)
]
for _, row in rule4.iterrows():
    alerts.append({
        'time': row['time'], 'rule': 'Anomalously Clean Authentication at Scale',
        'severity': 'MEDIUM', 'user': row['source_user'],
        'computer': row['destination_computer'],
        'detail': f"Velocity: {row['logins_per_hour']}, Destinations: {row['unique_destination_computers']}"
    })

alerts_df = pd.DataFrame(alerts)
alerts_df.to_csv('siem_alerts.csv', index=False)

print(f"FINAL SIEM RULE SET — Total alerts: {len(alerts_df):,}")
print(f"\nAlerts by rule:")
print(alerts_df['rule'].value_counts())

# Overall precision and recall
all_attack_alerts = alerts_df['user'].isin(ATTACK_ACCOUNTS)
overall_precision = all_attack_alerts.mean()
accounts_caught = len(set(alerts_df['user']) & ATTACK_ACCOUNTS)

print(f"\nOverall precision: {overall_precision*100:.2f}%")
print(f"Distinct attack accounts caught: {accounts_caught}/60")
print(f"Total alert volume reduction from original: {(1 - len(alerts_df)/35183)*100:.1f}%")

FINAL SIEM RULE SET — Total alerts: 16,972

Alerts by rule:
rule
Anomalously Clean Authentication at Scale    11243
Excessive Lateral Movement                    5564
Batch Scripted Authentication                  145
Suspicious Remote Logon Type                    20
Name: count, dtype: int64

Overall precision: 73.23%
Distinct attack accounts caught: 44/60
Total alert volume reduction from original: 51.8%


In [10]:
from sklearn.metrics import confusion_matrix

df['siem_flagged'] = df['time'].isin(alerts_df['time']) & df['source_user'].isin(alerts_df['user'])

# More precise: build flagged set from exact (time, user) pairs that fired
flagged_pairs = set(zip(alerts_df['time'], alerts_df['user']))
df['siem_flagged'] = df.apply(lambda r: (r['time'], r['source_user']) in flagged_pairs, axis=1).astype(int)

y_true = df['is_attack'].values
y_siem = df['siem_flagged'].values

tn, fp, fn, tp = confusion_matrix(y_true, y_siem).ravel()
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
f1        = 2*precision*recall/(precision+recall) if (precision+recall) > 0 else 0
fpr       = fp / (fp + tn) if (fp + tn) > 0 else 0
accuracy  = (tp + tn) / (tp + tn + fp + fn)

print("=" * 55)
print("  RQ1 — SIEM CORRELATION RULES: FORMAL METRICS (LANL)")
print("=" * 55)
print(f"  True Positives  (TP): {tp:,}")
print(f"  False Positives (FP): {fp:,}")
print(f"  True Negatives  (TN): {tn:,}")
print(f"  False Negatives (FN): {fn:,}")
print()
print(f"  Accuracy  : {accuracy:.4f}")
print(f"  Precision : {precision:.4f}")
print(f"  Recall    : {recall:.4f}")
print(f"  F1-Score  : {f1:.4f}")
print(f"  FPR       : {fpr:.4f}")
print("=" * 55)

pd.DataFrame([{
    'system': 'SIEM', 'accuracy': round(accuracy,4),
    'precision': round(precision,4), 'recall': round(recall,4),
    'f1': round(f1,4), 'fpr': round(fpr,4),
    'alerts': len(alerts_df), 'tp': int(tp), 'fp': int(fp),
    'tn': int(tn), 'fn': int(fn)
}]).to_csv('siem_metrics.csv', index=False)
print("\nSaved siem_metrics.csv")

  RQ1 — SIEM CORRELATION RULES: FORMAL METRICS (LANL)
  True Positives  (TP): 7,746
  False Positives (FP): 4,183
  True Negatives  (TN): 53,817
  False Negatives (FN): 2,475

  Accuracy  : 0.9024
  Precision : 0.6493
  Recall    : 0.7579
  F1-Score  : 0.6994
  FPR       : 0.0721

Saved siem_metrics.csv


In [1]:
# ─────────────────────────────────────────────────────────────
# 03 SIEM RULES — User-Day Level Evaluation (REBUILT)
# Rules fire at event level but are evaluated at user-day level
# matching the Isolation Forest's detection granularity
# ─────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
from sklearn.metrics import (precision_score, recall_score,
                              f1_score, confusion_matrix)

# Load raw events and aggregated user-day data
df_raw = pd.read_csv('parsed_logs.csv')
df_agg = pd.read_csv('df_with_features.csv')
df_raw['day'] = (df_raw['time'] // 86400).astype(int)

ATTACK_ACCOUNTS = set(df_raw[df_raw['is_attack']==1]['source_user'].unique())
print(f"Raw events: {len(df_raw):,}")
print(f"User-day rows: {len(df_agg):,}")
print(f"Attack accounts: {len(ATTACK_ACCOUNTS)}")

# ── Apply 4 SIEM rules at event level ────────────────────────
# Rule 1: Excessive Lateral Movement (T1021)
# Threshold from aggregated feature — user touches >= 10 destinations in a day
rule1_users = set(
    df_agg[df_agg['unique_destinations'] >= 10]['source_user']
)

# Rule 2: Suspicious Remote Logon Type (T1021.001)
rule2_users = set(
    df_raw[df_raw['logon_type'] == 'RemoteInteractive']['source_user']
)

# Rule 3: Batch/Scripted Authentication (T1059)
rule3_users = set(
    df_raw[df_raw['logon_type'] == 'Batch']['source_user']
)

# Rule 4: Anomalously Clean High-Volume Authentication (T1078)
# Zero failures AND high total logons AND broad destination reach
rule4_users = set(
    df_agg[
        (df_agg['failure_rate'] == 0) &
        (df_agg['total_logons'] >= 10) &
        (df_agg['unique_destinations'] >= 5)
    ]['source_user']
)

print(f"\nRule firing counts (distinct users flagged):")
print(f"  Rule 1 — Excessive Lateral Movement:  {len(rule1_users):,} users")
print(f"  Rule 2 — Suspicious Remote Logon:     {len(rule2_users):,} users")
print(f"  Rule 3 — Batch Scripted Auth:         {len(rule3_users):,} users")
print(f"  Rule 4 — Clean High-Volume Auth:      {len(rule4_users):,} users")

# ── Evaluate at user-day level ────────────────────────────────
# A user-day is flagged if the user fired any rule
all_flagged_users = rule1_users | rule2_users | rule3_users | rule4_users

df_agg['siem_flagged'] = df_agg['source_user'].isin(all_flagged_users).astype(int)
df_agg['rule1_flagged'] = df_agg['source_user'].isin(rule1_users).astype(int)
df_agg['rule2_flagged'] = df_agg['source_user'].isin(rule2_users).astype(int)
df_agg['rule3_flagged'] = df_agg['source_user'].isin(rule3_users).astype(int)
df_agg['rule4_flagged'] = df_agg['source_user'].isin(rule4_users).astype(int)

y_true = df_agg['is_attack'].values
y_siem = df_agg['siem_flagged'].values

tn, fp, fn, tp = confusion_matrix(y_true, y_siem).ravel()
precision = tp / (tp+fp) if (tp+fp) > 0 else 0
recall    = tp / (tp+fn) if (tp+fn) > 0 else 0
f1        = 2*precision*recall/(precision+recall) if (precision+recall) > 0 else 0
fpr       = fp / (fp+tn) if (fp+tn) > 0 else 0

print(f"\n{'='*55}")
print(f"  SIEM RULES — User-Day Level Evaluation (RQ1)")
print(f"{'='*55}")
print(f"  True Positives:  {tp:,}")
print(f"  False Positives: {fp:,}")
print(f"  True Negatives:  {tn:,}")
print(f"  False Negatives: {fn:,}")
print(f"\n  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1:        {f1:.4f}")
print(f"  FPR:       {fpr:.4f}")
print(f"  Alert Volume (user-days): {y_siem.sum():,}")

# Save
pd.DataFrame([{
    'system': 'SIEM', 'precision': round(precision,4),
    'recall': round(recall,4), 'f1': round(f1,4),
    'fpr': round(fpr,4), 'alerts': int(y_siem.sum()),
    'tp': int(tp), 'fp': int(fp), 'tn': int(tn), 'fn': int(fn)
}]).to_csv('siem_metrics.csv', index=False)

df_agg.to_csv('df_with_features.csv', index=False)
print(f"\nSaved siem_metrics.csv")

Raw events: 68,221
User-day rows: 14,416
Attack accounts: 60

Rule firing counts (distinct users flagged):
  Rule 1 — Excessive Lateral Movement:  69 users
  Rule 2 — Suspicious Remote Logon:     17 users
  Rule 3 — Batch Scripted Auth:         26 users
  Rule 4 — Clean High-Volume Auth:      196 users

  SIEM RULES — User-Day Level Evaluation (RQ1)
  True Positives:  149
  False Positives: 175
  True Negatives:  14,074
  False Negatives: 18

  Precision: 0.4599
  Recall:    0.8922
  F1:        0.6069
  FPR:       0.0123
  Alert Volume (user-days): 324

Saved siem_metrics.csv
